# Split scanned page on the longest vertical dark line

Pipeline:
1. Find the longest near-vertical dark line (allowed to be slightly curved / not perfectly orthogonal, but mostly uninterrupted).
2. Whiten that line in the image (without bleeding into nearby letters).
3. Cut the image along the (curved) line. The two halves get padded with `#E01AD1` so each half is a rectangle whose missing area is the parallelogram-ish wedge between the curve and the straight vertical.
4. Stack the right half **below** the left half and save the result.

The detection algorithm lives in `line_detection.py` so the GUI tool (`correction_tool.py`) and this notebook stay in sync.

## Manual corrections (optional)

Two JSON files in the source folder, both produced by `correction_tool.py`:

- **`starts.json`** -- per-image seed x. Marked `autogen: True` if produced automatically, `autogen: False` if the user adjusted it in the GUI.
- **`overlays.json`** -- per-image full traced curve (`xs[y]` array, plus `y0`, `y1`). Same `autogen` flag.

**Precedence rules:**

| starts entry            | overlays entry          | Notebook does                              |
|-------------------------|-------------------------|--------------------------------------------|
| --                      | --                      | Auto-detect from scratch                   |
| autogen / manual        | --                      | Re-trace from the given seed_x             |
| --                      | autogen / manual        | Use the precomputed overlay as-is          |
| autogen                 | autogen                 | Use the precomputed overlay (consistent)   |
| manual                  | autogen                 | **Manual wins** -- re-trace from seed_x    |
| any                     | manual                  | Use the precomputed (manual) overlay       |

When the notebook auto-detects (no JSON inputs), it writes BOTH JSON files with `autogen: True` for every image so the GUI can pick up where the notebook left off.

In [ ]:
import json
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import line_detection as ld
from line_detection import (
    DEFAULT_CONFIG,
    find_vertical_line,
    trace_from_seed,
    trace_from_seed_refined,
    hex_to_rgb,
    load_json_safely,
    save_json,
    xs_to_serializable,
    xs_from_serializable,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Tunable parameters (mirrors line_detection.DEFAULT_CONFIG, plus a
# few rendering knobs that are notebook-only).
CONFIG = dict(DEFAULT_CONFIG)
CONFIG.update({
    'debug': True,
    'save_debug_dir': None,
})
PAD_RGB = hex_to_rgb(CONFIG['pad_color_hex'])
PAD_RGB

In [ ]:
# Step 2: whiten the line, avoiding nearby letters
def whiten_line(img_rgb, xs, y0, y1, cfg=CONFIG):
    out = img_rgb.copy()
    H, W, _ = out.shape
    chan = ld.to_single_channel(img_rgb, cfg.get('gray_mode', 'gray'))
    dark = chan <= cfg['dark_threshold']
    hw = cfg['whiten_half_width']
    guard = cfg['letter_guard_distance']
    for y in range(y0, y1+1):
        x = xs[y]
        if x < 0:
            continue
        left_lo = max(0, x - hw - guard); left_hi = max(0, x - hw)
        right_lo = min(W, x + hw + 1);     right_hi = min(W, x + hw + 1 + guard)
        letter_left = dark[y, left_lo:left_hi].any() if left_hi > left_lo else False
        letter_right = dark[y, right_lo:right_hi].any() if right_hi > right_lo else False
        local_hw = 0 if (letter_left or letter_right) else hw
        lo = max(0, x - local_hw); hi = min(W, x + local_hw + 1)
        out[y, lo:hi] = (255, 255, 255)
    return out


In [ ]:
# Step 3 + 4: split into two parallelogram-padded rectangles, stack vertically
def split_on_curve(img_rgb, xs, y0, y1, pad_rgb=PAD_RGB):
    H, W, _ = img_rgb.shape
    detected = xs[y0:y1+1] if y1 >= y0 else np.array([], dtype=np.int32)
    detected = detected[detected >= 0]
    if len(detected) == 0:
        raise ValueError('No line detected')
    fallback_x = int(np.median(detected))
    full_xs = np.where(xs >= 0, xs, fallback_x)
    left_w = int(full_xs.max())
    right_origin = int(full_xs.min())
    right_w = W - right_origin
    left = np.full((H, left_w, 3), pad_rgb, dtype=np.uint8)
    right = np.full((H, right_w, 3), pad_rgb, dtype=np.uint8)
    for y in range(H):
        cut = int(full_xs[y])
        if cut > 0:
            left[y, :cut] = img_rgb[y, :cut]
        if cut < W:
            dst_start = max(0, cut - right_origin)
            right[y, dst_start:dst_start + (W - cut)] = img_rgb[y, cut:]
    return left, right

def stack_left_over_right(left, right, pad_rgb=PAD_RGB):
    target_w = max(left.shape[1], right.shape[1])
    def pad_right_to(img, w):
        if img.shape[1] == w:
            return img
        pad = np.full((img.shape[0], w - img.shape[1], 3), pad_rgb, dtype=np.uint8)
        return np.concatenate([img, pad], axis=1)
    return np.concatenate([pad_right_to(left, target_w),
                           pad_right_to(right, target_w)], axis=0)


In [ ]:
# Resolve which (xs, y0, y1, seed_x) to use for a given image, given
# optional precomputed JSON entries. Implements the precedence table.
def resolve_line_for_image(img_rgb, fname, starts_entries, overlays_entries, cfg):
    s_entry = starts_entries.get(fname)
    o_entry = overlays_entries.get(fname)
    s_manual = s_entry is not None and not s_entry.get('autogen', True)
    o_manual = o_entry is not None and not o_entry.get('autogen', True)

    if o_manual:
        xs = xs_from_serializable(o_entry['xs'])
        y0 = int(o_entry['y0']); y1 = int(o_entry['y1'])
        seed_x = int(s_entry['seed_x']) if s_entry else int(np.median(xs[xs >= 0]))
        return xs, y0, y1, seed_x, 'overlay-cache (manual)'

    if s_manual:
        seed_x = int(s_entry['seed_x'])
        xs, (y0, y1) = trace_from_seed_refined(img_rgb, seed_x, cfg)
        return xs, y0, y1, seed_x, 'retrace-from-manual-seed'

    if o_entry is not None:
        xs = xs_from_serializable(o_entry['xs'])
        y0 = int(o_entry['y0']); y1 = int(o_entry['y1'])
        seed_x = int(s_entry['seed_x']) if s_entry else int(np.median(xs[xs >= 0]))
        return xs, y0, y1, seed_x, 'overlay-cache (autogen)'

    if s_entry is not None:
        seed_x = int(s_entry['seed_x'])
        xs, (y0, y1) = trace_from_seed_refined(img_rgb, seed_x, cfg)
        return xs, y0, y1, seed_x, 'retrace-from-autogen-seed'

    xs, (y0, y1), seed_x = find_vertical_line(img_rgb, cfg)
    return xs, y0, y1, seed_x, 'auto'


In [ ]:
def make_overlay(img_rgb, xs, y0, y1):
    dbg = img_rgb.copy()
    H, W, _ = dbg.shape
    if y1 >= y0:
        for y in range(y0, y1+1):
            x = xs[y]
            if x >= 0:
                dbg[y, max(0, x-2):min(W, x+3)] = (255, 0, 0)
    return dbg

def process_image(in_path, out_path, cfg=CONFIG,
                  starts_entries=None, overlays_entries=None):
    img = np.array(Image.open(in_path).convert('RGB'))
    fname = Path(in_path).name
    starts_entries = starts_entries or {}
    overlays_entries = overlays_entries or {}

    xs, y0, y1, seed_x, source = resolve_line_for_image(
        img, fname, starts_entries, overlays_entries, cfg)

    if y1 < y0:
        raise RuntimeError(f'No vertical line in {in_path} (source={source})')

    if cfg.get('debug'):
        plt.figure(figsize=(8, 10))
        plt.imshow(make_overlay(img, xs, y0, y1))
        plt.title(f'{fname}: rows {y0}-{y1}/{img.shape[0]}, seed x={seed_x}, src={source}')
        plt.axis('off')
        plt.show()

    if cfg.get('save_debug_dir'):
        ddir = Path(cfg['save_debug_dir'])
        ddir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(make_overlay(img, xs, y0, y1)).save(
            ddir / (Path(in_path).stem + '_overlay.png'))

    whitened = whiten_line(img, xs, y0, y1, cfg)
    left, right = split_on_curve(whitened, xs, y0, y1)
    stacked = stack_left_over_right(left, right)

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(stacked).save(out_path)
    return {'out_path': out_path, 'xs': xs, 'y0': y0, 'y1': y1,
            'seed_x': seed_x, 'source': source}


In [ ]:
# Single-image quick test:
# r = process_image('input_pages/gold_00016.jpg', 'output_pages/gold_00016_split.png')
# print(r['source'], r['seed_x'], r['y0'], r['y1'])

# Batch process a folder

In [ ]:
SOURCE_DIR = Path('input_pages')
TARGET_DIR = Path('output_pages')
DEBUG_DIR  = Path('output_pages/_debug')
EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

# JSON files live alongside the source images so the GUI tool finds them too.
STARTS_JSON   = SOURCE_DIR / 'starts.json'
OVERLAYS_JSON = SOURCE_DIR / 'overlays.json'

batch_cfg = dict(CONFIG)
batch_cfg['debug'] = False
batch_cfg['save_debug_dir'] = str(DEBUG_DIR)
# For brown-paper editions, switch to blue channel:
# batch_cfg['gray_mode'] = 'blue'
# batch_cfg['dark_threshold'] = 50

TARGET_DIR.mkdir(parents=True, exist_ok=True)

starts_in   = load_json_safely(STARTS_JSON)
overlays_in = load_json_safely(OVERLAYS_JSON)
starts_entries   = starts_in.get('entries', {})
overlays_entries = overlays_in.get('entries', {})
print(f'Loaded {len(starts_entries)} starts entries, {len(overlays_entries)} overlay entries')

files = sorted(p for p in SOURCE_DIR.iterdir() if p.suffix.lower() in EXTS)
print(f'Found {len(files)} image(s) in {SOURCE_DIR}')

starts_out_entries = {}
overlays_out_entries = {}
errors = []

for i, src in enumerate(files, 1):
    dst = TARGET_DIR / f'{src.stem}_split.png'
    try:
        r = process_image(src, dst, batch_cfg,
                          starts_entries=starts_entries,
                          overlays_entries=overlays_entries)
        starts_existing = starts_entries.get(src.name)
        overlays_existing = overlays_entries.get(src.name)
        seed_autogen = (starts_existing is None or starts_existing.get('autogen', True))
        overlay_autogen = (overlays_existing is None or overlays_existing.get('autogen', True))
        if r['source'].startswith('retrace'):
            overlay_autogen = True
        if r['source'].startswith('overlay-cache') and overlays_existing is not None:
            overlay_autogen = bool(overlays_existing.get('autogen', True))

        starts_out_entries[src.name] = {
            'seed_x': int(r['seed_x']),
            'autogen': bool(seed_autogen),
        }
        overlays_out_entries[src.name] = {
            'y0': int(r['y0']),
            'y1': int(r['y1']),
            'xs': xs_to_serializable(r['xs']),
            'autogen': bool(overlay_autogen),
        }
        print(f'[{i}/{len(files)}] OK  {src.name}: src={r["source"]:30s} seed_x={r["seed_x"]} rows={r["y0"]}-{r["y1"]}')
    except Exception as e:
        errors.append((src, e))
        print(f'[{i}/{len(files)}] ERR {src.name}: {e}')

save_json(STARTS_JSON, {'config_used': dict(DEFAULT_CONFIG), 'entries': starts_out_entries})
save_json(OVERLAYS_JSON, {'config_used': dict(DEFAULT_CONFIG), 'entries': overlays_out_entries})

print(f'\nDone. {len(errors)} error(s).')
print(f'Wrote {STARTS_JSON} ({len(starts_out_entries)} entries)')
print(f'Wrote {OVERLAYS_JSON} ({len(overlays_out_entries)} entries)')
print(f'Overlays in {DEBUG_DIR}/')
for s, e in errors:
    print(' -', s.name, '->', e)